# Stage 1 - Phases 2-6: forensic branch (F1..F5)One notebook runs every forensic model. Change `MODEL_NAME` below and re-runtop to bottom.Recommended execution order (implementation numbering and experiment orderdiffer on purpose - the screen-recapture-specific branches are tried before themore generic CDC):```bayar_resnet18  ->  chromaticity  ->  frequency  ->  lcdf  ->  cdc   F1 (Phase 2)     F3 (Phase 3)     F4 (Phase 4)  F5 (Ph.5) F2 (Phase 6)```| key | model | reference || --- | --- | --- || `bayar_resnet18` | constrained convolution + ResNet18 | Bayar and Stamm, IEEE TIFS 2018 || `chromaticity` | CMA-inspired chromaticity branch | Chen et al., CVPR 2024 || `frequency` | FMAG / M2FM-inspired frequency branch | Chen et al., TIFS 2024 / TDSC 2025 || `lcdf` | LC&DF-inspired dual stream (main candidate) | Li et al., IEEE WIFS 2025 || `cdc` | central difference convolution classifier | Yu et al., CVPR 2020 |The pipeline order is fixed and deliberate:```fixed video-level split      -> native-resolution decoded frame      -> native-resolution 256x256 crop      -> forensic model      -> patch probability      -> frame aggregation -> video aggregation      -> Stage 1 Macro-F1 + threshold search```A frame is **never** resized before cropping: that would destroy the moire,aliasing and sub-pixel traces these models detect. Scoring is always at videolevel, never at patch level.

## 1. Setup

In [ ]:
from __future__ import annotationsimport sysfrom pathlib import Pathimport numpy as npimport pandas as pdimport torchimport yaml# The package is expected to be installed with `pip install -e .` from the# repository root. The fallback keeps a fresh clone usable without installing.try:    import blackbox_detection  # noqa: F401except ModuleNotFoundError:    _root = Path.cwd()    while _root != _root.parent and not (_root / "pyproject.toml").is_file():        _root = _root.parent    sys.path.insert(0, str(_root / "src"))from blackbox_detection.utils import seed_everything, setup_loggerprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
import jsonfrom blackbox_detection.stage1 import (    AggregationConfig,    PatchAugmentConfig,    Stage1Evaluator,    Stage1ForensicDataset,    Stage1Trainer,    TrainConfig,    ValidationSubsetSpec,    apply_split,    build_dataloader,    build_forensic_samplers,    build_forensic_transforms,    build_stage1_model,    build_validation_subsets,    forensic_batch_adapter,    load_manifest,    load_split,    save_predictions,    search_best_threshold,)from blackbox_detection.stage1.evaluator import probabilities_to_labelsfrom blackbox_detection.utils import load_checkpoint, stage1_scorelogger = setup_logger("stage1.forensic")

## 2. Paths

In [ ]:
REPO_ROOT = Path.cwd()while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():    REPO_ROOT = REPO_ROOT.parentCONFIG_DIR = REPO_ROOT / "configs" / "stage1"OUTPUT_ROOT = REPO_ROOT / "outputs" / "stage1"OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)print("repo   :", REPO_ROOT)print("configs:", CONFIG_DIR)print("outputs:", OUTPUT_ROOT)

## 3. Config`MODEL_NAME` is the only thing to change between forensic experiments;per-model settings live in `configs/stage1/forensic.yaml`.

In [ ]:
# One of: bayar_resnet18 | chromaticity | frequency | lcdf | cdcMODEL_NAME = "bayar_resnet18"FORENSIC_CONFIG = yaml.safe_load((CONFIG_DIR / "forensic.yaml").read_text(encoding="utf-8"))if MODEL_NAME not in FORENSIC_CONFIG["models"]:    raise ValueError(        f"{MODEL_NAME!r} has no entry in configs/stage1/forensic.yaml; "        f"available: {sorted(FORENSIC_CONFIG['models'])}"    )def merge_config(defaults: dict, overrides: dict) -> dict:    """Shallow-merge per-model overrides onto the shared defaults."""    merged = {key: dict(value) for key, value in defaults.items()}    for section, values in overrides.items():        merged.setdefault(section, {})        merged[section] = {**merged[section], **values}    return mergedCONFIG = merge_config(FORENSIC_CONFIG["defaults"], FORENSIC_CONFIG["models"][MODEL_NAME])ADAPTER = forensic_batch_adapter()SEED = int(CONFIG["train"]["seed"])PATCH_SIZE = int(CONFIG["data"]["patch_size"])RUN_DIR = OUTPUT_ROOT / MODEL_NAMERUN_DIR.mkdir(parents=True, exist_ok=True)seed_everything(SEED, deterministic=False)print(MODEL_NAME, "->", RUN_DIR)print(json.dumps(CONFIG["model"], indent=2))

## 4. Data### 4.1 Fixed manifest and split

In [ ]:
DATA_CONFIG = yaml.safe_load((CONFIG_DIR / "dlc2021.yaml").read_text(encoding="utf-8"))MANIFEST_PATH = REPO_ROOT / DATA_CONFIG["paths"]["manifest"]SPLIT_PATH = REPO_ROOT / DATA_CONFIG["paths"]["split"]manifest = load_manifest(MANIFEST_PATH, check_paths_exist=False)split = load_split(SPLIT_PATH)splits = apply_split(manifest, split)train_manifest, val_manifest = splits["train"], splits["val"]assert not set(train_manifest["video_id"]) & set(val_manifest["video_id"])print("train videos:", len(train_manifest), "| val videos:", len(val_manifest))print(train_manifest["label"].value_counts().to_dict(), val_manifest["label"].value_counts().to_dict())

In [ ]:
subset_specs = [    ValidationSubsetSpec(        name=spec["name"],        column=spec["column"],        minimum=spec.get("minimum"),        maximum=spec.get("maximum"),        min_per_class=int(spec.get("min_per_class", 20)),    )    for spec in DATA_CONFIG.get("validation_subsets", [])]# VAL-A is the whole stratified validation set; the specs above are VAL-B style# controlled diagnostics, reported only when they hold enough videos per class.VAL_SUBSETS = {"val_a": val_manifest["video_id"].tolist()}for name, result in build_validation_subsets(val_manifest, subset_specs).items():    if result.usable:        VAL_SUBSETS[name] = result.video_ids    else:        print(f"skipping {name}: {result.reason}")print("evaluation subsets:", {name: len(ids) for name, ids in VAL_SUBSETS.items()})

### 4.2 Native-resolution frames and patchesTraining draws random frames and random patch positions; validation usesdeterministic frame and patch positions so scores are comparable across epochsand models.

In [ ]:
data_config = CONFIG["data"]augmentation_config = CONFIG["augmentation"]patch_augment = PatchAugmentConfig(    hflip_prob=float(augmentation_config["hflip_prob"]),    vflip_prob=float(augmentation_config["vflip_prob"]),    rot90_prob=float(augmentation_config["rot90_prob"]),    spectral_augment_prob=float(augmentation_config["spectral_augment_prob"]),    spectral_augment_alpha_range=tuple(augmentation_config["spectral_augment_alpha_range"]),    spectral_augment_beta_std=float(augmentation_config["spectral_augment_beta_std"]),    spectral_augment_keep_outside=bool(augmentation_config["spectral_augment_keep_outside"]),)train_transform, val_transform = build_forensic_transforms(train_config=patch_augment)train_frame_sampler, train_patch_sampler = build_forensic_samplers(    train=True,    num_frames=int(data_config["train_num_frames"]),    num_patches=int(data_config["train_num_patches"]),    patch_size=PATCH_SIZE,)val_frame_sampler, val_patch_sampler = build_forensic_samplers(    train=False,    num_frames=int(data_config["val_num_frames"]),    num_patches=int(data_config["val_num_patches"]),    patch_size=PATCH_SIZE,    val_grid=int(data_config["val_patch_grid"]),)train_dataset = Stage1ForensicDataset(    train_manifest,    frame_sampler=train_frame_sampler,    patch_sampler=train_patch_sampler,    transform=train_transform,    patch_size=PATCH_SIZE,    on_error="zero",    deterministic=False,)val_dataset = Stage1ForensicDataset(    val_manifest,    frame_sampler=val_frame_sampler,    patch_sampler=val_patch_sampler,    transform=val_transform,    patch_size=PATCH_SIZE,    on_error="zero",    deterministic=True,)train_loader = build_dataloader(    train_dataset,    batch_size=int(data_config["batch_size"]),    shuffle=True,    num_workers=int(data_config["num_workers"]),    seed=SEED,    drop_last=True,)val_loader = build_dataloader(    val_dataset,    batch_size=int(data_config["val_batch_size"]),    shuffle=False,    num_workers=int(data_config["num_workers"]),    seed=SEED,)print("patches per training video:", train_dataset.num_units)print("patches per validation video:", val_dataset.num_units)batch = next(iter(train_loader))adapted = ADAPTER.unpack(batch, "cpu")print("patch batch:", tuple(batch["patches"].shape), "-> units:", tuple(adapted.inputs.shape))print("frames sampled in this batch:", sorted(set(batch["frame_indices"].reshape(-1).tolist()))[:10])

In [ ]:
# Deterministic validation sampling must be stable across passes, otherwise# per-epoch scores are not comparable.first = val_dataset[0]second = val_dataset[0]assert torch.equal(first["patches"], second["patches"])assert first["frame_indices"].tolist() == second["frame_indices"].tolist()print("deterministic validation patches confirmed")# Patch values must reach the model as linear RGB in [0, 1]: the chromaticity# and spectrum representations are computed from unmodified pixel values.print("patch range:", float(first["patches"].min()), "-", float(first["patches"].max()))

## 5. Model

In [ ]:
model = build_stage1_model(    MODEL_NAME,    finetune_mode=CONFIG["model"]["finetune_mode"],    unfreeze_last_n=int(CONFIG["model"]["unfreeze_last_n"]),    **CONFIG["model"]["params"],)from blackbox_detection.stage1.models import count_parametersprint("blocks:", len(model.blocks), "| feature dim:", model.feature_dim)print("parameters:", count_parameters(model))print("preprocessing:", dict(model.preprocessing()))with torch.no_grad():    probe = model(adapted.inputs[:2])print("logits:", tuple(probe.shape))

### 5.1 Inspect the forensic representationSanity-check what the model actually sees. This is where a silently brokenrepresentation shows up.

In [ ]:
with torch.no_grad():    sample = adapted.inputs[:2]    if hasattr(model, "chromaticity"):        chroma = model.chromaticity(sample)        print("chromaticity map:", tuple(chroma.shape),              "| finite:", bool(torch.isfinite(chroma).all()),              "| mean %.4f std %.4f" % (float(chroma.mean()), float(chroma.std())))    if hasattr(model, "spectrum"):        spectrum = model.spectrum(sample)        print("log spectrum:", tuple(spectrum.shape),              "| mean %.4f std %.4f" % (float(spectrum.mean()), float(spectrum.std())))    if hasattr(model, "residual"):        residual = model.residual(sample)        weight = model.bayar.constrained_weight.reshape(model.bayar.out_channels, 3, -1)        centre = weight.shape[-1] // 2        print("Bayar residual:", tuple(residual.shape),              "| centre weight:", float(weight[..., centre].mean()),              "| non-centre sum:", float((weight.sum(-1) - weight[..., centre]).mean()))    if hasattr(model, "stream_features"):        streams = model.stream_features(sample)        print("LC&DF streams:", {name: tuple(value.shape) for name, value in streams.items()})        if hasattr(model.df_representation, "alpha"):            print("FMAG alpha:", float(model.df_representation.alpha))

## 6. Training

In [ ]:
train_config = CONFIG["train"]trainer_config = TrainConfig(    epochs=int(train_config["epochs"]),    learning_rate=float(train_config["learning_rate"]),    head_learning_rate=(        float(train_config["head_learning_rate"])        if train_config.get("head_learning_rate") is not None        else None    ),    weight_decay=float(train_config["weight_decay"]),    warmup_ratio=float(train_config["warmup_ratio"]),    grad_accum_steps=int(train_config["grad_accum_steps"]),    max_grad_norm=float(train_config["max_grad_norm"]),    amp=bool(train_config["amp"]),    label_smoothing=float(train_config.get("label_smoothing", 0.0)),    early_stopping_patience=int(train_config["early_stopping_patience"]),    eval_every=int(train_config["eval_every"]),    seed=SEED,    output_dir=RUN_DIR,    model_name=MODEL_NAME,    wandb_enabled=False,)trainer = Stage1Trainer(    model,    trainer_config,    adapter=ADAPTER,    aggregation=AggregationConfig(        frame_method=CONFIG["evaluation"]["aggregation"]["frame_method"],        video_method=CONFIG["evaluation"]["aggregation"]["video_method"],    ),    model_config={"name": MODEL_NAME, "params": CONFIG["model"]["params"]},)print("device:", trainer.device, "| amp:", trainer.amp)outcome = trainer.fit(train_loader, val_loader, subsets=VAL_SUBSETS)print()print(f"best epoch {outcome.best_epoch}: Macro-F1 {outcome.best_macro_f1:.4f} "      f"at threshold {outcome.best_threshold:.3f}")

## 7. ValidationPatch probabilities are aggregated to frames, then to videos, and only thenscored with the official metric.

In [ ]:
load_checkpoint(RUN_DIR / "best.pt", model=model, map_location=trainer.device,                restore_rng_state=False)evaluator = Stage1Evaluator(    model,    ADAPTER,    device=trainer.device,    amp=trainer.amp,    aggregation=trainer.aggregation,)result, units = evaluator.evaluate(val_loader, subsets=VAL_SUBSETS, return_units=True)print(f"Macro-F1            : {result.macro_f1:.4f}  (official Stage 1 metric)")print(f"Macro-F1 @ thr 0.5  : {result.macro_f1_at_default:.4f}")print(f"optimal threshold   : {result.threshold:.4f}")print(f"class-wise F1       : {result.per_class_f1}")print(f"per-dataset Macro-F1: {result.dataset_scores}")print(f"videos              : {result.num_videos} ({result.num_invalid_videos} with decode problems)")print()for name, payload in result.subset_scores.items():    print(f"{name}: {payload}")

### 7.1 Aggregation choiceMean probability is the initial choice. Compare the registered alternatives before changing it.

In [ ]:
from blackbox_detection.stage1.evaluator import (    aggregate_unit_predictions,    evaluate_predictions,)rows = []for video_method in ("mean", "median", "trimmed_mean", "logit_mean", "max"):    aggregated = aggregate_unit_predictions(        units, aggregation=AggregationConfig(frame_method="mean", video_method=video_method)    )    scored = evaluate_predictions(aggregated)    rows.append(        {            "frame": "mean",            "video": video_method,            "macro_f1": scored.macro_f1,            "threshold": scored.threshold,        }    )display(pd.DataFrame(rows))

### 7.2 Threshold search

In [ ]:
labels = result.predictions["label"].tolist()probabilities = result.predictions["prob_rerecorded"].to_numpy()sweep = pd.DataFrame({"threshold": np.round(np.arange(0.05, 1.0, 0.05), 2)})sweep["macro_f1"] = [    stage1_score(labels, probabilities_to_labels(probabilities, threshold))    for threshold in sweep["threshold"]]display(sweep.set_index("threshold").T)best_threshold, best_score = search_best_threshold(labels, probabilities)print(f"searched threshold {best_threshold:.4f} -> Macro-F1 {best_score:.4f}")

## 8. Results

In [ ]:
display(outcome.history)print("Validation predictions (video level):")display(result.predictions.head(10))print("Patch-score spread per video (aggregation sanity):")display(    units.groupby("video_id")["prob_rerecorded"]    .agg(["count", "mean", "std", "min", "max"])    .head(10))

### 8.1 How to read these numbersA very high DLC-2021 Macro-F1 can mean the model latched onto document layout,text, borders, source resolution or a specific display/camera pair rather than arecapture trace. Record per run: Macro-F1, class-wise F1, the optimal threshold,overfitting behaviour, VAL-A versus the controlled VAL-B subset, and predictiondiversity against the other models (notebook 04).A wide patch-score spread inside one video is not automatically bad: recaptureartefacts are locally uneven. It does mean the aggregation choice matters, whichis what section 7.1 measures.

## 9. Save

In [ ]:
save_predictions(result.predictions, RUN_DIR / "val_predictions.csv")outcome.history.to_csv(RUN_DIR / "history.csv", index=False)units.to_csv(RUN_DIR / "val_unit_predictions.csv", index=False)summary = {    "model_name": MODEL_NAME,    "val_macro_f1": float(result.macro_f1),    "val_macro_f1_at_0.5": float(result.macro_f1_at_default),    "best_threshold": float(result.threshold),    "per_class_f1": result.per_class_f1,    "best_epoch": int(outcome.best_epoch),    "num_val_videos": int(result.num_videos),    "subset_scores": result.subset_scores,    "preprocessing": dict(model.preprocessing()),    "patch_size": PATCH_SIZE,    "units_per_val_video": int(val_dataset.num_units),}(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")print("artefacts in", RUN_DIR)for path in sorted(RUN_DIR.iterdir()):    print("  ", path.name)print()print("Change MODEL_NAME and re-run for the next forensic model, then go to "      "scripts/stage1/04_compare_and_fuse.ipynb")